# CrewAI — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-2/crewai.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-2/crewai.ipynb)

Companion to: [CrewAI](https://anacodicai.org/learn/ai-agents/6-2/crewai)


## What you'll build

The **real `crewai` library** — not a simulation. We construct a real
`Agent`, a real `Task`, and a real `Crew` and inspect exactly what CrewAI
built from them: roles, goals, the default LLM config each agent picked up,
and the sequential process graph connecting tasks together.

**No API key needed for any of this** — creating agents, tasks, and a crew
never calls a model. Only the very last, clearly-marked bonus cell
(`crew.kickoff()`) needs a real key, because that's the one line that
actually asks an LLM to do the work. `pip install crewai` if running locally.

### Setup

Two environment quirks CrewAI runs into outside a plain laptop/Colab setup —
both copy-pasteable fixes:

1. CrewAI uses `chromadb` for memory, which needs a newer `sqlite3` than some
   systems ship with. `RuntimeError: Your system has an unsupported version
   of sqlite3` is fixed by swapping in `pysqlite3-binary`'s bundled sqlite3
   before crewai (transitively) imports the real one.
2. CrewAI stores task-output history in a local SQLite file under your home
   directory. If that home directory lives on a network filesystem (common
   on shared servers and HPC clusters), SQLite's WAL locking fails with
   `Database initialization error: locking protocol`. Pointing
   `XDG_DATA_HOME` at a local, non-network directory fixes it — harmless on
   a normal machine or Colab, where the default is already local.

In [1]:
# %pip install -q crewai pysqlite3-binary
import os
import sys
import tempfile

os.environ.setdefault("XDG_DATA_HOME", tempfile.gettempdir())

try:
    import pysqlite3
    sys.modules["sqlite3"] = pysqlite3
except ImportError:
    pass  # only needed on systems with sqlite3 < 3.35

import crewai
from crewai import Agent, Task, Crew, Process

print("real crewai", crewai.__version__, "imported")

real crewai 1.15.14 imported


## Step 1 — a real Agent

`role`, `goal`, and `backstory` aren't just labels for a printout — CrewAI
folds them into the system prompt it builds for whichever LLM ends up
running this agent. Notice `allow_delegation=False`: this agent can't hand
work off to teammates, which matters once a crew has more than one.

In [2]:
grader = Agent(
    role="Grading Assistant",
    goal="Evaluate a short-answer response against a rubric, fairly and consistently",
    backstory=(
        "An experienced teaching assistant who always checks every rubric "
        "criterion explicitly before assigning a grade, and never assigns a "
        "grade without a one-sentence justification."
    ),
    allow_delegation=False,
    verbose=False,
)

print("role:     ", grader.role)
print("goal:     ", grader.goal)
print("delegates:", grader.allow_delegation)

role:      Grading Assistant
goal:      Evaluate a short-answer response against a rubric, fairly and consistently
delegates: False


## Step 2 — every agent already has a real, inspectable LLM config

We never set an LLM. CrewAI picked a real default (`OpenAICompletion`,
`gpt-4.1-mini`) and attached it to the agent — with `api_key=None`, because
nothing has asked it to actually call out yet. Constructing an agent and
*running* an agent are two separate steps in CrewAI, same as LangGraph
separates compiling a graph from invoking it.

In [3]:
print(type(grader.llm).__name__)
print("model:  ", grader.llm.model)
print("api_key:", grader.llm.api_key)

assert grader.llm.api_key is None
print("\nassert passed: the agent has a real LLM object attached, unauthenticated")

OpenAICompletion
model:   gpt-4.1-mini
api_key: None

assert passed: the agent has a real LLM object attached, unauthenticated


## Step 3 — a real Task, bound to that agent

`{answer}` is a template slot, same idea as LangChain's `{topic}` — CrewAI
fills it in from the inputs passed to `kickoff()`.

In [4]:
grading_task = Task(
    description=(
        "Grade the following short-answer response against the rubric below. "
        "Rubric: award full credit only if the response names both the cause "
        "and the effect. Response to grade: {answer}"
    ),
    expected_output="A letter grade (A-F) and exactly one sentence of justification.",
    agent=grader,
)

print("assigned agent:", grading_task.agent.role)
print("expected output:", grading_task.expected_output)

assigned agent: Grading Assistant
expected output: A letter grade (A-F) and exactly one sentence of justification.


## Step 4 — a Crew: agents plus tasks plus a process

`Process.sequential` means CrewAI runs tasks in the order given, each one
seeing the previous task's output. With only one task here it's trivial, but
this is the exact same object a 5-agent, 8-task pipeline would use.

In [5]:
crew = Crew(
    agents=[grader],
    tasks=[grading_task],
    process=Process.sequential,
    verbose=False,
)

print("agents in crew:", [a.role for a in crew.agents])
print("tasks in crew: ", len(crew.tasks))
print("process:       ", crew.process)

assert crew.process == Process.sequential
assert len(crew.agents) == 1 and len(crew.tasks) == 1
print("\nasserts passed: a real Crew object was constructed from a real Agent and Task")

agents in crew: ['Grading Assistant']
tasks in crew:  1
process:        Process.sequential

asserts passed: a real Crew object was constructed from a real Agent and Task


## Step 5 — a second agent, and delegation

Add a `Reviewer` agent that CAN delegate, and a second task that depends on
the first agent's output. This is the shape CrewAI actually exists for:
multiple specialized agents cooperating on a pipeline, not one agent alone.

In [6]:
reviewer = Agent(
    role="Grade Reviewer",
    goal="Spot-check grades assigned by the Grading Assistant for consistency",
    backstory="A senior TA who reviews a sample of grades before they're released.",
    allow_delegation=True,
    verbose=False,
)

review_task = Task(
    description="Review the grade and justification produced for: {answer}",
    expected_output="APPROVE or FLAG, plus a one-sentence reason.",
    agent=reviewer,
)

two_agent_crew = Crew(
    agents=[grader, reviewer],
    tasks=[grading_task, review_task],
    process=Process.sequential,
    verbose=False,
)

print("agents:", [a.role for a in two_agent_crew.agents])
print("reviewer can delegate:", reviewer.allow_delegation)
assert len(two_agent_crew.agents) == 2
print("\nassert passed: a two-agent sequential crew was constructed")

agents: ['Grading Assistant', 'Grade Reviewer']
reviewer can delegate: True

assert passed: a two-agent sequential crew was constructed


## Bonus (needs a real API key) — actually run it

Everything above builds and inspects the real object model with zero network
calls. This cell is the one line that needs a key — set `OPENAI_API_KEY` (or
configure `grader.llm` for a different provider) and uncomment it to see the
grader actually produce a graded response.

In [7]:
import os

if os.environ.get("OPENAI_API_KEY"):
    result = crew.kickoff(inputs={"answer": "It rained, so the ground got wet."})
    print(result)
else:
    print("Set OPENAI_API_KEY and re-run this cell to see crew.kickoff() "
          "actually call the LLM. Skipped here since this notebook promises "
          "to run with no key required.")

Set OPENAI_API_KEY and re-run this cell to see crew.kickoff() actually call the LLM. Skipped here since this notebook promises to run with no key required.


## Try it yourself

- Change `allow_delegation=True` on `grader` and add a third agent it could
  delegate to — inspect how CrewAI's task assignment changes.
- Swap `Process.sequential` for `Process.hierarchical` (needs a `manager_llm`)
  and compare what changes about how `crew.tasks` gets executed.
- Set `grader.llm.model = "gpt-4o"` (or point it at a different provider
  entirely) before `kickoff()` — notice the Agent/Task/Crew code above
  doesn't change at all, same as swapping the model in the LangChain notebook.